# Bronze Layer


### Create Catalog

In [0]:
--create catalog if not exists capstone;

### -Use Catalog
### -Create Schema

In [0]:
use catalog capstone;

CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

See if we have any created _Volumes_

In [0]:
SHOW VOLUMES IN capstone.bronze;

If we don't have any then we go ahead and create the needed _Volume_

In [0]:
CREATE VOLUME IF NOT EXISTS capstone.bronze.raw_files;

Test volume

### Ingest all tables in the bronze layers

In [0]:
%python
from pyspark.sql.functions import current_timestamp, lit

spark.sql("USE CATALOG capstone")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

base_path = "/Volumes/capstone/bronze/raw_files"

datasets = [
    {
        "folder_name": "diseases",
        "file_name": "diseases.csv",
        "table_name": "diseases",
        "source_system": "DISEASES_RAW"
    },
    {
        "folder_name": "medications",
        "file_name": "medications.csv",
        "table_name": "medications",
        "source_system": "MEDICATIONS_RAW"
    },
    {
        "folder_name": "pharmacies",
        "file_name": "pharmacies.csv",
        "table_name": "pharmacies",
        "source_system": "PHARMACIES_RAW"
    },
    {
        "folder_name": "inventory",
        "file_name": "inventory.csv",
        "table_name": "inventory",
        "source_system": "INVENTORY_RAW"
    },
    {
        "folder_name": "Insurance",
        "file_name": "insurance.csv",
        "table_name": "insurance",
        "source_system": "INSURANCE_RAW"
    },
    {
        "folder_name": "geography",
        "file_name": "geography.csv",
        "table_name": "geography",
        "source_system": "GEOGRAPHY_RAW"
    }
]

def load_bronze_dataset(folder_name: str, file_name: str, table_name: str, source_system: str):
    path = f"{base_path}/{folder_name}/{file_name}"

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(path)
    )

    df = (
        df
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_system", lit(source_system))
    )

    (
        df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(f"capstone.bronze.{table_name}")
    )

    print(f"Loaded: capstone.bronze.{table_name} from {path}")


for d in datasets:
    load_bronze_dataset(
        folder_name=d["folder_name"],
        file_name=d["file_name"],
        table_name=d["table_name"],
        source_system=d["source_system"]
    )